# Workspaces

A workspace is a folder with `epilens.toml`, paper and index paths, saved metadata, and optional task definitions. Notebook 08 constructed runtime settings directly. Here the workspace provides those settings and lets the CLI discover them from the current directory.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb
WORK_DIR = nb.bootstrap("epilens-workspace-")


## Create and inspect a workspace

`create_workspace` is the Python equivalent of `epilens init`. It writes local storage defaults and creates folders in temporary scratch space. Keep credentials in the environment rather than `epilens.toml`.


In [ ]:
from epilens.workspace import create_workspace, find_workspace, load_workspace

workspace = create_workspace(WORK_DIR / "flu-review", name="flu-review")
print("Root:", workspace.root)
print("Contents:", [entry.name for entry in sorted(workspace.root.iterdir())])
print(workspace.config_path.read_text())


## Discover the folder, then configure the runtime

`find_workspace` searches upward for `epilens.toml`; `load_workspace` opens a specified folder. Its local paths and backend choices become a `RuntimeConfig` for the service layer. The printed Qdrant URL is an optional setting; the default `index_backend = "file"` keeps this workspace local.


In [ ]:
from epilens.services.studio import runtime_config_for_workspace

print("Found from papers/:", find_workspace(workspace.root / "papers").name)
print("Loaded explicitly:", load_workspace(workspace.root).name)

runtime_config = runtime_config_for_workspace(workspace)
print("Index:", runtime_config.index_dir.relative_to(workspace.root))
print("Metadata:", runtime_config.metadata_backup.relative_to(workspace.root))


## Add a task to the workspace

A JSON `TaskSpec` in `tasks/` becomes selectable by key without editing Python. This miner spec yields an item list for verification; a classifier task would instead define a fixed label set.


In [ ]:
import json
from epilens.services.studio import TaskService

funding_spec = {
    "key": "find_funding_sources",
    "kind": "miner",
    "label": "Funding sources",
    "retrieval_templates": ["Who funded the study, and which grants are named?"],
}
(workspace.root / "tasks" / "find_funding_sources.json").write_text(
    json.dumps(funding_spec, indent=2)
)
tasks = TaskService(workspace)
print("Workspace tasks:", [spec.key for spec in tasks.specs()])


The equivalent CLI path is `epilens init <path>`, place papers in `papers/`, run `epilens index`, then run a task from inside the workspace. `epilens tasks validate --task-file <file>` checks a custom spec before use. The workspace supplies paths and task definitions; the runtime from notebook 08 performs retrieval and workflows.
